In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")



/var/folders/4m/z36lvyp57r37vs_gdh88c2mc0000gn/T/ipykernel_3277/2875600212.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name="BAAI/bge-large-en-v1.5")
/Users/arunkumar/anaconda3/envs/py312/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

file=TextLoader("speech.txt")
doc=file.load()

In [4]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import euclidean_distances

Text=[" this is USA and it is huge in trade",
           "Trump is the president of USA",
            "Modi is the prime minister of INDIA" ]

doc_embeding=embeddings.embed_documents(Text)
print(doc_embeding)
query = "who is Donald"
query_embedding = embeddings.embed_query(query)





[[0.0025112966541200876, 0.041719235479831696, 0.01675150729715824, 0.06758780032396317, -0.014011360704898834, 0.011701833456754684, 0.01682320050895214, 0.008481659926474094, 0.0636761263012886, 0.02817600592970848, 0.010128103196620941, -0.025400297716259956, -0.007310320157557726, -0.03790596127510071, 0.0031718250829726458, -0.006264050491154194, -0.0015269014984369278, 0.007415491621941328, -0.007512885611504316, 0.034932222217321396, -0.02433507889509201, -0.01102185808122158, -0.042682331055402756, -0.07136102765798569, -0.005272351671010256, 0.012514326721429825, 0.020798569545149803, 0.051366329193115234, 0.0520191565155983, 0.07851757854223251, -0.023519644513726234, -0.010272659361362457, 0.042108919471502304, -0.03202439472079277, -0.02743242308497429, 0.02194477804005146, 0.04127245023846626, -0.014249082654714584, 0.024453839287161827, -0.05037889629602432, -0.007682041265070438, -0.029174312949180603, 0.04239681735634804, 0.030993569642305374, -0.023239368572831154, 0.0

In [5]:
cos_sim = cosine_similarity([query_embedding], doc_embeding)
print("Cosine Similarity:", cos_sim)
#To dispaly the text of the vector
best_idx = cos_sim.argmax()
print("Most relevant text:", Text[best_idx])

Cosine Similarity: [[0.41607929 0.57777485 0.44272236]]
Most relevant text: Trump is the president of USA


In [6]:
#do index using any similarity search

import faiss
from langchain_community.vectorstores import FAISS 
from langchain_community.docstore.in_memory import InMemoryDocstore

#do index using any similarity search COSINE,L2, etc
index=faiss.IndexFlatL2(384)

In [7]:
vector_store=FAISS(
embedding_function=embeddings, 
index=index, 
docstore=InMemoryDocstore(),
index_to_docstore_id={},
)

print(vector_store)


In [8]:
texts = ["AI is", "USA is power", "Dog circuit"]
vector_store = FAISS.from_texts(texts, embeddings)

#single document no [] list is required
vector_store.similarity_search("tell me about AI",k=2)

[Document(id='ab9469d6-aad7-4612-b3d7-e22cf3e7ca16', metadata={}, page_content='AI is'),
 Document(id='af134493-ebf4-46bc-b789-c014591c8b0c', metadata={}, page_content='Dog circuit')]

🧠 Why UUIDs?
Useful for uniquely identifying documents in vector stores

Helps with updates, deletions, or logging

You can persist UUIDs in a DB alongside your vector index

In [9]:
from langchain.schema import Document
import uuid
docs = [
    Document(page_content=text, metadata={"uuid": str(uuid.uuid4())})
    for text in texts
]
vector_stores = FAISS.from_documents(docs, embeddings)

for doc_id, doc in vector_store.docstore._dict.items():
    print("Doc ID:", doc_id)
    print("Text:", doc.page_content)
    print("UUID:", doc.metadata.get("uuid"))

Doc ID: ab9469d6-aad7-4612-b3d7-e22cf3e7ca16
Text: AI is
UUID: None
Doc ID: bf2c756e-075c-4ebd-b3a1-ecfc1f8c5542
Text: USA is power
UUID: None
Doc ID: af134493-ebf4-46bc-b789-c014591c8b0c
Text: Dog circuit
UUID: None


In [10]:
vector_store.index_to_docstore_id

{0: 'ab9469d6-aad7-4612-b3d7-e22cf3e7ca16',
 1: 'bf2c756e-075c-4ebd-b3a1-ecfc1f8c5542',
 2: 'af134493-ebf4-46bc-b789-c014591c8b0c'}

| Feature               | `Flat`                | `IVF` (Inverted File Index)        | `HNSW` (Graph-based Index)          |
| --------------------- | --------------------- | ---------------------------------- | ----------------------------------- |
| Type of Search     | Exact                 | Approximate (cluster-based)        | Approximate (graph-based traversal) |
| Speed               | Slow (linear scan)    | Fast (search only in top clusters) | Very Fast (graph walk)              |


| Dataset Size              | Recommended Index                 |
| ------------------------- | --------------------------------- |
| UPTO 1L                     | `IndexFlatL2` or `IndexFlatIP`    |
| UPTO 1M                  | `IndexIVFFlat` or `IndexHNSWFlat` |
| > 1M                      | `IndexIVFPQ` or `IndexHNSWFlat`   |


In [11]:
# from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="c",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [12]:
index=faiss.IndexFlatIP(1024) # using FlatIP inner product for similarity search it is going to apply cosine

#used 1024 flat indes and it differes according to various embedding models
#created vectore store
vector_store1=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},

)


In [13]:
vector_store1.add_documents(documents=documents)

['1a36d30f-9e2d-4473-a4f1-fd67467a5f58',
 'c55e8a9b-40d6-442c-85bc-dd77628132b6',
 '4063964f-6007-41e2-a5e8-9164143c45c4',
 'c3f05f5b-6a3e-455a-b9b0-f2a8a529c9f1',
 '3b689527-14e6-4c98-8929-7419500c48f2',
 '4608873b-f82f-44de-b28b-77245a183c53',
 '7777efe6-31a2-4abc-80cb-b6258b1d661d',
 '45645a65-675b-4896-9efa-a8873ad34d66',
 '18f70a54-f94a-4696-8b43-c413d1f7c20b',
 '029a22d4-8996-4a10-b946-2cdc22cb12a5']

In [14]:
#need to give some querry metadata
# no embedding is done, because it is alredy been taken care by Langchain vector store where we defined the embedding

result=vector_store1.similarity_search("LangGraph is the best framework for building stateful, agentic applications!",
                                 k=2,
                                 filter={"source":{"$eq":"website"}}
                                )

In [15]:
#it shows the information of the data
print(result[0].metadata)
result[0].page_content

{'source': 'website'}


'The top 10 soccer players in the world right now.'

In [16]:
#inside the vector store we have retrival option inside the FAISS
retriever=vector_store1.as_retriever(search_kargs={"k":3})

# the retrival is done in Memory and we can also sotre the result to store it in local
retriever.invoke("LangGraph is the best framework")

[Document(id='4063964f-6007-41e2-a5e8-9164143c45c4', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='45645a65-675b-4896-9efa-a8873ad34d66', metadata={'source': 'tweet'}, page_content='c'),
 Document(id='029a22d4-8996-4a10-b946-2cdc22cb12a5', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='7777efe6-31a2-4abc-80cb-b6258b1d661d', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

In [17]:
vector_store1.save_local("FIASS INDEX to store in local")

#upto this Inmemory and Local memory is discussed
#Folow pinecone for cloud

In [18]:
#Load the loacal store Index, if we stop server we can also take it from Local when it is down

new_vector_search=FAISS.load_local("FIASS INDEX to store in local",embeddings,allow_dangerous_deserialization=True)

#note we cannot use invoke, we need to use similarity_search
new_vector_search.similarity_search("LangGraph is the best framework",k=2)

[Document(id='4063964f-6007-41e2-a5e8-9164143c45c4', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='45645a65-675b-4896-9efa-a8873ad34d66', metadata={'source': 'tweet'}, page_content='c')]

BUILDING RAG shortly

In [19]:

from langchain_community.document_loaders import PyPDFLoader

file_path=r"/Users/arunkumar/Agentic_2.0/Krish_copy/2.4-VectorDatabase/FAISS/llama2.pdf"

loader=PyPDFLoader(file_path)

In [20]:
print(len(loader.load()))
pages=loader.load()

#pages=[]  #AZYLODER IS HASTER
#asunc for pages in loader.alazy_load():
    #pages.append(page)

#divide data Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

#declaration of splitter
splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50) 

77


In [21]:
#spliting the documents

split_docs=splitter.split_documents(pages)

#there will be multiple Documents created divided pages into chunks in every chunk we have 500 character
print(split_docs)

[Document(metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/arunkumar/Agentic_2.0/Krish_copy/2.4-VectorDatabase/FAISS/llama2.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}, page_content='Llama 2: Open Foundation and Fine-Tuned Chat Models\nHugo Touvron∗ Louis Martin† Kevin Stone†\nPeter Albert Amjad Almahairi Yasmine Babaei Nikolay Bashlykov Soumya Batra\nPrajjwal Bhargava Shruti Bhosale Dan Bikel Lukas Blecher Cristian Canton Ferrer Moya Chen\nGuillem Cucurull David Esiobu Jude Fernandes Jeremy Fu Wenyin Fu Brian Fuller\nCynthia Gao Vedanuj Goswami Naman Goyal Anthony Hartshorn Saghar Hosseini Rui Hou\nHakan Inan Marcin Kardas Viktor Kerkez Madian Khabsa Isabel Klouma

In [24]:
#new vector store creation
# will create a retriver on top of it, need to keep how many documets we need to fetch from vector database

index=faiss.IndexFlatL2(1024)
vector_store2=FAISS(
embedding_function=embeddings, 
index=index, 
docstore=InMemoryDocstore(),
index_to_docstore_id={},
)

vector_store2.add_documents(documents=split_docs)

# will create a retriver on top of it, need to keep how many documets we need to fetch from vector database
retriver=vector_store2.as_retriever(
    search_kwargs={"k":10}  # hyperparameter, multiple expriments
)


In [54]:
vector_store2.add_documents(documents=split_docs)

['06a60eda-645d-49d4-872e-3efac17e36b3',
 '3a59c4eb-556b-4709-bb03-bb00c9e74bcf',
 '7a1c30ba-a556-4a69-9aee-98ce06e2bd93',
 'c1d1d76e-6ae2-4da1-8a9c-e2c8f0105059',
 'fabdff1f-ec0a-4837-983c-968c713cbfdf',
 'cd0d4713-c43e-4658-a23e-1c8777386905',
 '1f27dc04-9e53-4c78-b73e-37cec296fbfe',
 '45a5b702-af88-4347-967a-c5499a42d01c',
 'a9afc03f-0698-4a47-b317-745c56568a59',
 'd07432c8-3d98-4f80-b7dd-1dadfe047f3e',
 '8b13882e-214b-488d-9eed-4a284248066f',
 '5a3c124d-8218-4e46-a7df-7b69b5f60a9a',
 '4bb19063-bc76-4df9-b3b7-edbe5d705d45',
 '92715f7a-27dd-40e1-ba9b-d809fc874ca2',
 '328ab8d1-0a19-452a-ab9d-fbe5acafc2b9',
 'da6be90f-ce09-412e-8db5-03976b9c7e34',
 '0a337b3e-ab29-4d9a-94d4-cf66a3554701',
 '8fca916b-92b5-40fc-8035-c40038d1d72e',
 '1972ed77-d2ec-4d60-b4ed-5b4898d4dcad',
 '2e34afed-f0d5-46fc-8354-cb170a3e0fc0',
 'bc92ea7e-65f8-426c-a397-3332f09e2f78',
 'e66abcbf-edac-4106-8512-59ceddbd99e0',
 '224aa203-9695-4d79-9c90-8b823ecb044e',
 'cc530794-d071-4a01-a207-6d19f8b61466',
 '11bf7f59-0e96-

In [55]:
retriver.invoke("what is lama model?")

#we havent completed RAG pipeline here, in vector DB we have page content and metadata

[Document(id='18954234-7080-47ce-8ecf-689bde136748', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': '/Users/arunkumar/Agentic_2.0/Krish_copy/2.4-VectorDatabase/FAISS/llama2.pdf', 'total_pages': 77, 'page': 76, 'page_label': '77'}, page_content='specific applications of the model. Please see the Responsible Use Guide available available at\nhttps://ai.meta.com/llama/responsible-user-guide\nTable 52: Model card forLlama 2.\n77'),
 Document(id='cd9518a4-1ad7-4420-bc03-2ea4d59a1b11', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner

VECTOR database is used as a retriver    querry --> Vector DB (can apply fillter)  --> back to LLM  -->O/P

vector DB is used to create the retriver PIPELINE
combining is RAG, Retriver from (querry ,vector)  , Generation(vector,LLM)

Basic RAG we can take from langchain hub (rag)

Flat Index , similarity search is done

In [58]:
import os 
from langchain_groq import ChatGroq

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=ChatGroq(model="qwen-qwq-32B")
gr_result=model.invoke("Hi my name is Arun")
print(gr_result.content)


#WE need to chain context,prompt,model,prompt


<think>
Okay, the user introduced themselves as Arun. I should respond politely. Let me see... Maybe say hello back and ask how I can assist them today. Keep it friendly and open-ended so they feel comfortable to ask anything. Let me make sure there's no typos and the tone is right. Alright, that should work.
</think>

Hello Arun! Nice to meet you. How can I assist you today? Feel free to ask me any questions or let me know if you need help with anything specific! 😊


In [62]:
from langchain import hub
prompt=hub.pull("rlm/rag-prompt")


import pprint #prettu print
pprint.pprint(prompt.messages)

#prompt for rag is alredy present can take from that

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]


[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]

In [83]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
# to take question on runtime we use Runnable pass through


#context(retriever).prompt(hub),model(google),parser(langchain)

#format_docs feteced page contect

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#from retriver contect it pass on throug fomrat_docs and then context and question it done by prompt

rag_chain=(
    {"context":retriver | format_docs ,"question":RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [82]:
rag_chain.invoke("what is lama model?")

"\n<think>\nOkay, the user is asking about the Llama model. Let me look through the provided context.\n\nThe context mentions Llama 2 specifically. It says it's a family of models with sizes from 7B to 70B parameters. They use architectures like auto-regressive transformers with techniques like SFT and RLHF for alignment. Also, Meta AI developed them, and they outperform previous versions and some other models.\n\nHmm, need to make sure I don't miss key points. The answer should include developer, variations in size, architecture, training period, and performance improvements. Keep it concise within three sentences. Don't mention unrelated parts like the license or responsible use guide unless necessary. Focus on the core details from the model card section.\n</think>\n\nThe Llama 2 model is a family of auto-regressive language models developed by Meta AI, available in sizes from 7B to 70B parameters. It uses supervised fine-tuning (SFT) and reinforcement learning with human feedback (

O/P:

"\n<think>\nOkay, the user is asking about the Llama model. Let me look through the provided context.\n\nThe context mentions Llama 2 specifically. It says it's a family of models with sizes from 7B to 70B parameters. They use architectures like auto-regressive transformers with techniques like SFT and RLHF for alignment. Also, Meta AI developed them, and they outperform previous versions and some other models.\n\nHmm, need to make sure I don't miss key points. The answer should include developer, variations in size, architecture, training period, and performance improvements. Keep it concise within three sentences. Don't mention unrelated parts like the license or responsible use guide unless necessary. Focus on the core details from the model card section.\n</think>\n\nThe Llama 2 model is a family of auto-regressive language models developed by Meta AI, available in sizes from 7B to 70B parameters. It uses supervised fine-tuning (SFT) and reinforcement learning with human feedback (RLHF) to align outputs with safety and helpfulness. Llama 2 outperforms its predecessor, Llama 1, and competitors like MPT and Falcon in various benchmarks."

suprviser, collabrative multi agent flow, runnable in savita chaneel